# CNN on MNIST — Approach and Analysis

This notebook covers the full workflow: data exploration, architecture design, training, and results analysis.

## 1. Data Exploration

MNIST is a dataset of handwritten digits (0–9) in grayscale.

| Property | Value |
|----------|-------|
| Training images | 60,000 |
| Test images | 10,000 |
| Dimensions | 28×28 pixels, 1 channel |
| Classes | 10 (digits 0 to 9) |
| Distribution | Balanced (~6,000 samples per class) |

The following statistics were computed from the training set via `scripts/00_explore_data.py`:
- **Mean**: 0.13066
- **Std**: 0.30811

These values are used directly in the preprocessing step (`transforms.Normalize((0.1307,), (0.3081,))`). Normalizing the pixel values centers the distribution around 0, which stabilizes training and speeds up convergence.

## 2. Preprocessing and Augmentation

**Preprocessing (train + test)**:
- `ToTensor()` — converts pixel values from `[0, 255]` to `[0.0, 1.0]`
- `Normalize((0.1307,), (0.3081,))` — centers and reduces variance

**Augmentation (train only)**:
- `RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1))` — random rotations, translations, and scaling

Augmentation is applied to the train set only. It makes the model more robust to geometric variations found in real-world data (slightly tilted or off-center digits). It also acts as an implicit regularizer: by making training examples artificially harder, it reduces the risk of overfitting.

## 3. Architecture — SimpleCNN

```
Input (1×28×28)
  → Conv2d(1→16, 3×3, padding=1) + ReLU + MaxPool(2×2)  →  16×14×14
  → Conv2d(16→32, 3×3, padding=1) + ReLU + MaxPool(2×2) →  32×7×7
  → Flatten                                              →  1568
  → Linear(1568→128) + ReLU
  → Linear(128→10)   [logits]
```

**Why this architecture?**

MNIST is a relatively simple problem (28×28 images, uniform background, low intra-class variability). A lightweight network is sufficient:
- 2 convolutional layers capture edges first, then more complex shapes
- MaxPool halves the spatial resolution at each step, reducing the number of parameters and enforcing translation invariance
- padding=1 preserves spatial dimensions after each convolution
- The FC layer (128 neurons) is sufficient to separate 10 classes

**Loss**: `CrossEntropyLoss` — standard for multi-class classification, combines LogSoftmax and NLLLoss.

**Optimizer**: `Adam` (lr=0.001) — adaptive, fast convergence without manual learning rate tuning.

## 4. Training Results

Configuration: 10 epochs, batch_size=32, lr=0.001, seed=42.

| Metric | Value |
|--------|-------|
| Train Loss (avg.) | 0.0880 |
| Test Loss (avg.) | 0.0360 |
| Test Accuracy (avg.) | **98.80%** |
| Test Accuracy (last epoch) | **99.11%** |

> Training curves are generated locally in `outputs/figures/` via `make train` (not versioned — unique timestamp per run).

## 5. Curve Analysis

**Convergence**: Train loss drops sharply between epoch 1 (~0.29) and epoch 2 (~0.08), then decreases slowly to ~0.05. Convergence is fast and stable — Adam with lr=0.001 is well-suited for this problem.

**No overfitting observed**: Test loss stays *below* train loss throughout training (~0.025 vs ~0.05 at the end). This is explained by **augmentation**: the train set is made deliberately harder (random rotations, translations, scaling), while the test set is evaluated on clean, normalized images. The model is penalized on augmented inputs but tested under ideal conditions — hence better performance on the test set.

**Accuracy**: It improves from 98.11% to 99.11% and stabilizes after epoch 4. Minor oscillations (±0.1%) reflect the stochastic variance of SGD over 10,000 samples.

**Conclusion**: The model generalizes well. Augmentation fulfilled its role as a regularizer. 10 epochs are sufficient to reach ~99% accuracy with this lightweight architecture — consistent with the state of the art for a simple CNN on MNIST.